# Vehicle Insurance Purchase Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `purchased`

## 2 · Project Overview

We predict whether a customer will **purchase vehicle insurance** based on demographics, credit score, vehicle age, and claim history.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a customer's age, income, vehicle age, prior claims, credit score, and region, predict whether they will purchase vehicle insurance.

## 5 · Why This Project Matters

- **Lead scoring** helps insurers target marketing spend.
- Understanding purchase drivers optimizes pricing and offers.
- Reduces cost of cold outreach.
- Directly impacts conversion rates and revenue.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,000 |
| **Features** | age, annual_income, vehicle_age_years, prev_claims, credit_score, region |
| **Target** | purchased (0 = no, 1 = yes) |
| **Class balance** | ~50/50 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "purchased"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: purchased
Save dir: E:\Github\Machine-Learning-Projects\Classification\Vehicle Insurance Purchase Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 7,000 customers with demographics and insurance purchase outcome.

In [4]:
np.random.seed(SEED)
n = 7000
age = np.random.randint(18, 70, n)
annual_income = np.round(np.random.lognormal(10.5, 0.6, n), 0)
vehicle_age_years = np.random.randint(0, 15, n)
prev_claims = np.random.poisson(0.8, n)
credit_score = np.random.randint(300, 850, n)
region = np.random.choice(["Urban", "Suburban", "Rural"], n, p=[0.4, 0.35, 0.25])
region_num = np.where(region == "Urban", 0, np.where(region == "Suburban", 1, 2))

score = (0.01 * age + 0.000005 * annual_income - 0.05 * vehicle_age_years
         - 0.1 * prev_claims + 0.002 * credit_score - 0.1 * region_num
         + np.random.normal(0, 0.7, n))
purchased = (score > np.median(score)).astype(int)

df = pd.DataFrame({"age": age, "annual_income": annual_income,
                    "vehicle_age_years": vehicle_age_years, "prev_claims": prev_claims,
                    "credit_score": credit_score, "region": region, "purchased": purchased})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['purchased'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (7000, 7)
Class balance:
purchased
1    0.5
0    0.5
Name: proportion, dtype: float64


,age,annual_income,vehicle_age_years,prev_claims,credit_score,region,purchased
0,56,35092.0,12,0,702,Urban,1
1,69,56398.0,7,2,487,Suburban,1
2,46,30275.0,8,1,792,Suburban,0
3,32,35580.0,13,0,554,Rural,0
4,60,50889.0,12,0,622,Suburban,1


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (7000, 7)

Missing values:
age                  0
annual_income        0
vehicle_age_years    0
prev_claims          0
credit_score         0
region               0
purchased            0
dtype: int64

Duplicate rows: 0

Dtypes:
age                    int32
annual_income        float64
vehicle_age_years      int32
prev_claims            int32
credit_score           int32
region                object
purchased              int64
dtype: object

Target 'purchased' confirmed. Value counts:
purchased
1    3500
0    3500
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = df.select_dtypes(include="number").columns.drop(TARGET).tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols[:6]):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Purchase Decision", fontsize=14)
plt.tight_layout()
plt.show()

corr = df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `purchased`.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind="bar", ax=axes[0], color=["salmon", "steelblue"], edgecolor="black")
axes[0].set_title("Purchase Distribution")
axes[0].set_xticklabels(["No (0)", "Yes (1)"], rotation=0)

df.groupby("region")[TARGET].mean().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Purchase Rate by Region")
axes[1].set_ylabel("Purchase Rate")
plt.tight_layout()
plt.show()
print(f"Overall purchase rate: {df[TARGET].mean():.1%}")

Overall purchase rate: 50.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (5600, 6), Test: (1400, 6)
Train class distribution:
purchased
1    0.5
0    0.5
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `region` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~50/50 by design.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["income_per_year_age"] = X_train["annual_income"] / (X_train["age"] + 1)
X_test["income_per_year_age"] = X_test["annual_income"] / (X_test["age"] + 1)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (7): ['age', 'annual_income', 'vehicle_age_years', 'prev_claims', 'credit_score', 'region', 'income_per_year_age']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))


BASELINE: Logistic Regression
Accuracy : 0.6900
F1       : 0.6851

              precision    recall  f1-score   support

           0       0.68      0.71      0.69       700
           1       0.70      0.67      0.69       700

    accuracy                           0.69      1400
   macro avg       0.69      0.69      0.69      1400
weighted avg       0.69      0.69      0.69      1400



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
LogisticRegression             0.690000           0.690000  0.759898  0.689923   0.690188  0.690000    0.018934
CalibratedClassifierCV         0.688571           0.688571  0.760222  0.688508   0.688725  0.688571    0.048401
LinearDiscriminantAnalysis     0.688571           0.688571  0.760298  0.688495   0.688758  0.688571    0.018493
LinearSVC                      0.688571           0.688571  0.760178  0.688508   0.688725  0.688571    0.019779
RidgeClassifier                0.688571           0.688571  0.760308  0.688495   0.688758  0.688571    0.021302
RidgeClassifierCV              0.688571           0.688571  0.760314  0.688495   0.688758  0.688571    0.021074
NearestCentroid                0.687143           0.687143  0.756339  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.6757
F1: 0.6657


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.6642  (1.4s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[73]	valid_0's binary_logloss: 0.595516
LightGBM F1: 0.6642  (1.1s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.6900  0.6851     0.6962  0.6743
FLAML                  0.6757  0.6657     0.6869  0.6457
LightGBM               0.6729  0.6642     0.6822  0.6471
CatBoost               0.6764  0.6642     0.6903  0.6400

Top 3 models: ['Logistic Regression', 'FLAML', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  Logistic Regression:
    Accuracy : 0.6900
    F1       : 0.6851
    Precision: 0.6962
    Recall   : 0.6743

  FLAML:
    Accuracy : 0.6757
    F1       : 0.6657
    Precision: 0.6869
    Recall   : 0.6457

  LightGBM:
    Accuracy : 0.6729
    F1       : 0.6642
    Precision: 0.6822
    Recall   : 0.6471


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.71      0.69       700
           1       0.70      0.67      0.69       700

    accuracy                           0.69      1400
   macro avg       0.69      0.69      0.69      1400
weighted avg       0.69      0.69      0.69      1400


Total errors: 434 / 1400 (31.0%)

Sample misclassifications:
      age  annual_income  vehicle_age_years  prev_claims  credit_score  region  income_per_year_age  actual  predicted  correct
4778   56        31008.0                  1            2           316     2.0           544.000000       1          0    False
4701   48        49900.0                  0            1           520     2.0          1018.367347       0          1    False
3879   60        19863.0                 10            0           500     1.0           325.622951       1          0    False
5092   19        36065.0                 11

## 25 · Interpretation and Business Insight

**Key findings:**
- **Credit score** and **income** are strong purchase predictors.
- **Previous claims** negatively affect insurance purchase.
- **Urban** customers have slightly higher purchase rates.

**Business takeaway:** Target high-credit-score, high-income prospects with newer vehicles for best conversion.

## 26 · Limitations

1. Synthetic data with limited realism.
2. No policy details (coverage types, premiums offered).
3. Missing behavioral data (website visits, quote requests).
4. No temporal patterns (seasonal buying).
5. Region is too coarse.

## 27 · How to Improve This Project

1. Use real insurance quote and conversion data.
2. Add quote details (premium amount, coverage options).
3. Include behavioral signals (time on site, pages viewed).
4. Add vehicle make/model/value as features.
5. Try uplift modeling to measure campaign impact.

## 28 · Production Considerations

- Integrate with CRM for real-time lead scoring.
- Output purchase probability for threshold-based targeting.
- A/B test different outreach strategies per risk segment.
- Retrain monthly with fresh conversion data.

## 29 · Common Mistakes

1. Not stratifying by region in analysis.
2. Including post-purchase data (leading to leakage).
3. Using accuracy on imbalanced data.
4. Ignoring income outliers.
5. Not considering cost of false positives (wasted marketing spend).

## 30 · Mini Challenge / Exercises

1. Remove `credit_score` and retrain — how much does F1 change?
2. Bin `age` into decades and compare purchase rates.
3. Create interaction: `vehicle_age_years * prev_claims` and test.
4. Plot ROC curve for the best model.
5. Try different thresholds and plot precision-recall tradeoff.

## 31 · Final Summary / Key Takeaways

1. **Credit score and income** are the strongest purchase predictors.
2. **Vehicle age and claims** add useful signal.
3. **Tree-based models** handle mixed features effectively.
4. **LazyPredict + FLAML** enable rapid screening.
5. **Lead scoring models** should output probabilities for targeting.
6. **Always validate on holdout data** to prevent overfitting.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Vehicle Insurance Purchase Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.6764,
    "f1": 0.6642,
    "precision": 0.6903,
    "recall": 0.64
  },
  "LightGBM": {
    "accuracy": 0.6729,
    "f1": 0.6642,
    "precision": 0.6822,
    "recall": 0.6471
  },
  "Logistic Regression": {
    "accuracy": 0.69,
    "f1": 0.6851,
    "precision": 0.6962,
    "recall": 0.6743
  },
  "FLAML": {
    "accuracy": 0.6757,
    "f1": 0.6657,
    "precision": 0.6869,
    "recall": 0.6457
  }
}
